In [1]:
import json
from pathlib import Path
from typing import List
import pandas as pd
import numpy as np

from model_ranking import (
    to_target_transfer_correlations,
    correlation_table,
    load_transfer_metric_results,
    dataframe_to_latex_table_styled,
)

INFO: P [MainThread] 2026-05-06 09:17:19,318 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def average_dataframes(dfs: List[pd.DataFrame]) -> pd.DataFrame:
    """Average a list of same-shape DataFrames element-wise, returning a table of the same shape."""
    avg_values = np.mean([df.values for df in dfs], axis=0)
    return pd.DataFrame(avg_values, index=dfs[0].index, columns=dfs[0].columns)


In [3]:
def average_targets(dfs: pd.DataFrame) -> pd.DataFrame:
    """Average dataframe across rows, returning a single-row dataframe."""
    return dfs.mean(axis=0).to_frame().T

# Mitochondria

In [26]:
performance_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/transfer_performance_scores.json"
with open(performance_path, "r") as f:
    performance_scores = json.load(f)["performance_scores"]

### Gauss EI

In [30]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/direct/EI_consistency/CMB"
selected_augmentations = ["a001-a003", "a003-a005", "a005-a007", "a007-a01", "a01-a012", "a012-a015", "a015-a02"]
#selected_augmentations = ["a003-a005", "a005-a007", "a007-a01", "a01-a012", "a012-a015", "a015-a02"]
#selected_augmentations = ["a01-a012"]
mito_gauss_EI_dfs: List[pd.DataFrame] = []
for aug in selected_augmentations:
    file_name = f"transfer_Gauss_{aug}_CMB_05f_05b_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    print(aug)
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores,
        )
    df_gauss_EI = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, num_sig_fig=3)
    print(df_gauss_EI)
    mito_gauss_EI_dfs.append(df_gauss_EI)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/direct/EI_consistency/CMB/transfer_Gauss_a001-a003_CMB_05f_05b_EI_scores.json
a001-a003
                 kt  kt pval  s rho  s rho pval     pr  pr pval
Task targets                                                   
Mito EPFL     0.733    0.002  0.879       0.002  0.820    0.000
     Hmito    0.657    0.002  0.786       0.002  0.751    0.001
     Rmito    0.766    0.002  0.897       0.002  0.868    0.000
     VNC      0.455    0.052  0.608       0.030  0.626    0.029
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency/direct/EI_consistency/CMB/transfer_Gauss_a003-a005_CMB_05f_05b_EI_scores.json
a003-a005
                 kt  kt pval  s rho  s rho pval     pr  pr pval
Task targets                                                   
Mito EPFL     0.771    0.002  0.904       

In [31]:
mito_per_target_avg_df = average_dataframes(mito_gauss_EI_dfs)
mito_per_target_avg_df

kt   kt pval     s rho  s rho pval        pr   pr pval
Task targets                                                              
Mito EPFL     0.804143  0.002000  0.922571    0.002000  0.877286  0.000000
     Hmito    0.722143  0.002000  0.854143    0.002286  0.810143  0.000143
     Rmito    0.834000  0.002000  0.928286    0.002000  0.855714  0.000000
     VNC      0.545429  0.016857  0.638000    0.031714  0.719857  0.010714

In [32]:
mito_avg_df = average_targets(mito_per_target_avg_df)
mito_avg_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
0,0.726429,0.005714,0.83575,0.0095,0.81575,0.002714


# Semantic nuclei

In [34]:
performance_path_nuclei = "/g/kreshuk/talks/consistency_results/patch_segmentation/nuclei/transfer_results/consistency/transfer_performance_F1_scores.json"
with open(performance_path_nuclei, "r") as f:
    performance_scores_nuclei = json.load(f)["performance_scores"]

In [35]:
base_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/nuclei/transfer_results/consistency/EI_consistency/FORG"
selected_augmentations = ["a0-005", "a005-01", "a01-02"]

nuclei_EI_gauss_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_Gauss_{aug}_Forg_EI_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores_nuclei,
    )
    df_avg_nuclei_gauss_EI = correlation_table(
        KT_scores, SP_scores, PE_scores, targets=targets, task="Nuclei", num_sig_fig=3
    )
    print(df_avg_nuclei_gauss_EI)
    nuclei_EI_gauss_dfs.append(df_avg_nuclei_gauss_EI)


Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/nuclei/transfer_results/consistency/EI_consistency/FORG/transfer_Gauss_a0-005_Forg_EI_scores.json
                     kt  kt pval  s rho  s rho pval     pr  pr pval
Task   targets                                                     
Nuclei BBBC039    0.619    0.060  0.714       0.086  0.995    0.000
       DSB2018    0.524    0.170  0.679       0.118  0.980    0.000
       Hoechst    0.619    0.076  0.750       0.062  0.954    0.001
       S_BIAD895  0.600    0.150  0.771       0.118  0.980    0.001
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/patch_segmentation/nuclei/transfer_results/consistency/EI_consistency/FORG/transfer_Gauss_a005-01_Forg_EI_scores.json
                     kt  kt pval  s rho  s rho pval     pr  pr pval
Task   targets                                                     
Nuclei BBBC039    0.619    0.068  0.786       0.058  0.996    0.000
     

In [36]:
nuclei_sem_per_target_avg_df = average_dataframes(nuclei_EI_gauss_dfs)
nuclei_sem_per_target_avg_df

kt   kt pval     s rho  s rho pval        pr   pr pval
Task   targets                                                                
Nuclei BBBC039    0.650667  0.055333  0.785667    0.062667  0.995667  0.000000
       DSB2018    0.524000  0.150000  0.679000    0.114000  0.988000  0.000000
       Hoechst    0.555667  0.122667  0.726000    0.076667  0.938333  0.002000
       S_BIAD895  0.644333  0.104667  0.790333    0.092000  0.982333  0.000667

In [37]:
nuclei_sem_avg_df = average_targets(nuclei_sem_per_target_avg_df)
nuclei_sem_avg_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
0,0.593667,0.108167,0.74525,0.086333,0.976083,0.000667


# Cells Instance

In [38]:
cells_performance_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/Cells/transfer_results/consistency/fullslice_No_Ignore/transfer_performance_MAP_scores.json"
with open(cells_performance_path, "r") as f:
    cells_performance_scores = json.load(f)["performance_scores"]

In [39]:
base_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/Cells/transfer_results/consistency/fullslice_No_Ignore/ARE_consis"
selected_augmentations = ["a001-005", "a005-01", "a01-015", "a015-02", "a02-025"]
#selected_augmentations = ["a02-025"]

cells_gauss_ARE_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_gauss_{aug}_Forg_ARE_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=cells_performance_scores,
        invert_transfer_metric=True,
    )
    df_cells_gauss_ARE = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, task='Cells')
    print(df_cells_gauss_ARE)
    cells_gauss_ARE_dfs.append(df_cells_gauss_ARE)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/Cells/transfer_results/consistency/fullslice_No_Ignore/ARE_consis/transfer_gauss_a001-005_Forg_ARE_scores.json
                 kt  kt pval  s rho  s rho pval    pr  pr pval
Task  targets                                                 
Cells FlyWing  0.64     0.04   0.79        0.04  0.79     0.02
      Ovules   0.43     0.16   0.60        0.14  0.95     0.00
      PNAS     0.93     0.00   0.98        0.00  0.83     0.01
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/Cells/transfer_results/consistency/fullslice_No_Ignore/ARE_consis/transfer_gauss_a005-01_Forg_ARE_scores.json
                 kt  kt pval  s rho  s rho pval    pr  pr pval
Task  targets                                                 
Cells FlyWing  0.57     0.06   0.76        0.05  0.91      0.0
      Ovules   0.43     0.18   0.57        0.17  0.87      0.0
      PNAS     0.71 

In [40]:
cells_per_target_avg_df = average_dataframes(cells_gauss_ARE_dfs)
cells_per_target_avg_df

kt  kt pval  s rho  s rho pval     pr  pr pval
Task  targets                                                   
Cells FlyWing  0.570    0.066  0.762       0.040  0.904    0.004
      Ovules   0.542    0.088  0.698       0.084  0.886    0.002
      PNAS     0.728    0.026  0.852       0.018  0.880    0.002

In [41]:
cells_avg_df = average_targets(cells_per_target_avg_df)
cells_avg_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
0,0.613333,0.06,0.770667,0.047333,0.89,0.002667


# Nuclei Instance

In [45]:
nuclei_performance_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/transfer_performance_MAP_scores.json"
with open(nuclei_performance_path, "r") as f:
    nuclei_performance_scores = json.load(f)["performance_scores"]

nuclei_performance_scores.pop("DSB2018", None)

{'BC_IN_model2': 0.4842800498008728,
 'HN_IN_model2': 0.5206860899925232,
 'Hst_IN_model3': 0.31693270802497864,
 '895_IN_model2': 0.43998968601226807,
 '1410_IN_model1': 0.49643415212631226}

In [46]:
base_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/ARE"
selected_augmentations = ["a0-005", "a005-01", "a01-02"]

nuclei_gauss_ARE_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_Gauss_{aug}_Forg_ARE_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    consistency_scores.pop("DSB2018", None)
    targets = list(consistency_scores.keys())
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=nuclei_performance_scores,
        invert_transfer_metric=True,
        )
    df_nuclei_gauss_ARE = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, task='Nuclei')
    print(df_nuclei_gauss_ARE)
    nuclei_gauss_ARE_dfs.append(df_nuclei_gauss_ARE)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/ARE/transfer_Gauss_a0-005_Forg_ARE_scores.json
                    kt  kt pval  s rho  s rho pval    pr  pr pval
Task   targets                                                   
Nuclei BBBC039    0.80     0.09    0.9        0.08  0.99     0.00
       Hoechst    0.80     0.08    0.9        0.07  0.85     0.07
       S_BIAD895  0.33     0.75    0.6        0.42  0.47     0.53
       S_BIAD634  0.80     0.07    0.9        0.09  0.93     0.02
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/ARE/transfer_Gauss_a005-01_Forg_ARE_scores.json
                    kt  kt pval  s rho  s rho pval    pr  pr pval
Task   targets                                                   
Nuclei BBBC039    0.20     0.86    0.5        0.50  0.72     0.17
       Hoechst    0.80     0.09    0.9        0

In [47]:
nuclei_inst_per_target_avg_df = average_dataframes(nuclei_gauss_ARE_dfs)
nuclei_inst_per_target_avg_df

kt   kt pval     s rho  s rho pval        pr   pr pval
Task   targets                                                                
Nuclei BBBC039    0.466667  0.476667  0.666667    0.326667  0.736667  0.186667
       Hoechst    0.866667  0.060000  0.933333    0.056667  0.890000  0.043333
       S_BIAD895  0.443333  0.610000  0.600000    0.500000  0.800000  0.200000
       S_BIAD634  0.866667  0.066667  0.933333    0.063333  0.840000  0.076667

In [49]:
nuclei_inst_avg_df = average_targets(nuclei_inst_per_target_avg_df)
nuclei_inst_avg_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
0,0.660833,0.303333,0.783333,0.236667,0.816667,0.126667


# COVID-IF Cells

In [21]:
performance_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/Covid_IF/transfer_results/transfer_performance_MSA_scores.json"
with open(performance_path, "r") as f:
    performance_scores = json.load(f)["performance_scores"]

In [22]:
base_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/Covid_IF/transfer_results/ARE"
selected_augmentations = ["a001-002", "a002-003", "a003-004", "a004-005"]

covid_if_gauss_ARE_dfs: List[pd.DataFrame] = []

for aug in selected_augmentations:
    file_name = f"transfer_gauss_{aug}_Forg_ARE_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    targets = list(consistency_scores.keys())

    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=performance_scores,
        invert_transfer_metric=True,
    )
    correlation_df = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets)
    print(aug)
    print(correlation_df)
    covid_if_gauss_ARE_dfs.append(correlation_df)

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/Covid_IF/transfer_results/ARE/transfer_gauss_a001-002_Forg_ARE_scores.json
a001-002
                kt  kt pval  s rho  s rho pval    pr  pr pval
Task targets                                                 
Mito Covid_IF  0.8     0.09    0.9        0.09  0.86     0.06
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/Covid_IF/transfer_results/ARE/transfer_gauss_a002-003_Forg_ARE_scores.json
a002-003
                kt  kt pval  s rho  s rho pval    pr  pr pval
Task targets                                                 
Mito Covid_IF  1.0     0.02    1.0        0.02  0.93     0.02
Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/Covid_IF/transfer_results/ARE/transfer_gauss_a003-004_Forg_ARE_scores.json
a003-004
                kt  kt pval  s rho  s rho pval    pr  pr pval
Task targets              

In [23]:
covid_if_per_target_avg_df = average_dataframes(covid_if_gauss_ARE_dfs)
covid_if_per_target_avg_df

,,kt,kt pval,s rho,s rho pval,pr,pr pval
Task,targets,,,,,,
Mito,Covid_IF,0.95,0.035,0.975,0.04,0.9075,0.0325


In [25]:
covid_if_avg_df = average_targets(covid_if_per_target_avg_df)
covid_if_avg_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
0,0.95,0.035,0.975,0.04,0.9075,0.0325


# ToothFairy

In [54]:
(0.9 + 0.9 + 0.93) / 3

0.91

In [55]:
toothfairy_avg_df = pd.DataFrame({
    "kt": [0.71],
    "kt pval": [0.04],
    "s rho": [0.8533],
    "s rho pval": [0.04],
    "pr": [0.91],
    "pr pval": [0.001]
    })
toothfairy_avg_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
0,0.71,0.04,0.8533,0.04,0.91,0.001


# Average Combined

In [56]:
avg_scores = [
    mito_avg_df,
    nuclei_sem_avg_df,
    cells_avg_df,
    nuclei_inst_avg_df,
    covid_if_avg_df,
    toothfairy_avg_df
]
names = ["mito", "nuclei_sem", "cells", "nuclei_inst", "covid_if", "toothfairy"]



In [58]:
from typing import Any
from pandas import Series


def combine_single_row_dataframes(dataframes: List[pd.DataFrame], names: List[str]) -> pd.DataFrame:
    """Stack single-row DataFrames into one DataFrame with `names` as row index."""
    if len(dataframes) != len(names):
        raise ValueError(
            f"Expected the same number of dataframes and names, got {len(dataframes)} and {len(names)}."
        )

    combined_rows: List[Series[Any]] = []
    for df, name in zip(dataframes, names):
        if df.shape[0] != 1:
            raise ValueError(
                f"Each dataframe must have exactly one row. '{name}' has shape {df.shape}."
            )
        row = df.iloc[0].copy()
        row.name = name
        combined_rows.append(row)

    return pd.DataFrame(combined_rows)

In [59]:
combined_avg_scores_df = combine_single_row_dataframes(avg_scores, names)
combined_avg_scores_df

,kt,kt pval,s rho,s rho pval,pr,pr pval
mito,0.726429,0.005714,0.835750,0.009500,0.815750,0.002714
nuclei_sem,0.593667,0.108167,0.745250,0.086333,0.976083,0.000667
cells,0.613333,0.060000,0.770667,0.047333,0.890000,0.002667
nuclei_inst,0.660833,0.303333,0.783333,0.236667,0.816667,0.126667
covid_if,0.950000,0.035000,0.975000,0.040000,0.907500,0.032500
toothfairy,0.710000,0.040000,0.853300,0.040000,0.910000,0.001000
